# Synthetic Preference Dataset Generation with Gemma Judge

## Overview

This notebook generates a preference dataset for training a reward model using:

1. **Source Data**: Sentences from `data/english_inputs.json` and `data/french_inputs.json`
2. **Translation Candidates**: For each source sentence, generate 2 translations using different sampling methods:
   - **Method 1**: High temperature (creative, diverse)
   - **Method 2**: Conservative (accurate, focused)
3. **Gemma as Judge**: Use the Gemma LLM to evaluate both translations and determine which is better
4. **Preference Pairs**: Store chosen/rejected pairs based on Gemma's judgment

## Output Files

- `en-ar-preferences.jsonl`: English→Arabic preference pairs
- `fr-ar-preferences.jsonl`: French→Arabic preference pairs
- `generation_stats.json`: Statistics on generation and judgments

## Key Features

- **Two sampling strategies** for diverse candidate generation
- **Gemma LLM judge** evaluates quality based on accuracy, fluency, completeness, and grammar
- **Checkpointing** to resume interrupted runs
- **Bilingual support** (English and French to Arabic)
- **Memory-optimized** batch processing

In [1]:
# ===========================
# PROJECT CONFIGURATION
# ===========================
import os
import torch
import random
import numpy as np
from pathlib import Path

# Detect project directory
_notebook_dir = Path.cwd()
_possible_paths = [
    Path("/home/aya/Desktop/Context_Specific_Machine_Translation_to_Arabic_Language/RLHF"),
    _notebook_dir,
    Path.cwd(),
]

PROJECT_DIR = None
for _path in _possible_paths:
    if (_path / "data").exists() or (_path / "0_config_setup.ipynb").exists():
        PROJECT_DIR = _path
        break

if PROJECT_DIR is None:
    PROJECT_DIR = _possible_paths[0]
    print(f"⚠️ Using default project path: {PROJECT_DIR}")
else:
    print(f"✅ Detected project directory: {PROJECT_DIR}")

DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / "models"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
LOGS_DIR = PROJECT_DIR / "logs"

for dir_path in [DATA_DIR, MODELS_DIR, OUTPUTS_DIR, LOGS_DIR]:
    dir_path.mkdir(exist_ok=True, parents=True)

# ===========================
# GPU CONFIGURATION
# ===========================
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"\nGPU Configuration:")
print(f"  GPUs available: {NUM_GPUS}")
if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

# ===========================
# HYPERPARAMETERS
# ===========================
SEED = 42
MAX_NEW_TOKENS = 128

# ===========================
# UTILITY FUNCTIONS
# ===========================
def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def format_translation_prompt(text, source_lang='en'):
    """Format input text as translation prompt"""
    lang_name = {'en': 'English', 'fr': 'French'}[source_lang]
    return f"Translate the following {lang_name} text to Arabic:\n\n{text}\n\nArabic translation:"

set_seed(SEED)
print("\n✅ Configuration loaded successfully!")


✅ Detected project directory: /home/aya/Desktop/Context_Specific_Machine_Translation_to_Arabic_Language/RLHF

GPU Configuration:
  GPUs available: 2
  GPU 0: NVIDIA GeForce RTX 5090
    Memory: 33.67 GB
  GPU 1: NVIDIA GeForce RTX 5090
    Memory: 33.67 GB

✅ Configuration loaded successfully!


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import json
import random
import gc
import os
import time
from pathlib import Path

set_seed(SEED)

print("Synthetic Preference Dataset Generation with Gemma Judge")
print("=" * 80)
print("Pipeline: Generate 2 candidates → Gemma judges → Store preferences")
print("=" * 80)


/home/aya/anaconda3/envs/pytorch_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Synthetic Preference Dataset Generation with Gemma Judge
Pipeline: Generate 2 candidates → Gemma judges → Store preferences


## Load SFT Model

In [3]:
# Clear GPU cache and set CUDA environment variables
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("GPU cache cleared")

# CUDA environment variables for optimized memory management
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ===========================
# MODEL LOADING CONFIGURATION
# ===========================
FORCE_CPU = False
USE_BFLOAT16 = True

print("\nModel Loading Configuration:")
print(f"Total GPUs: {NUM_GPUS}")
if NUM_GPUS > 0:
    print(f"Total VRAM: {NUM_GPUS * 31.36:.2f}GB")
print(f"SFT Model: 28.9B parameters")
print(f"Quantization: 8-bit")
print(f"Precision: {'bfloat16' if USE_BFLOAT16 else 'float32'}")


GPU cache cleared

Model Loading Configuration:
Total GPUs: 2
Total VRAM: 62.72GB
SFT Model: 28.9B parameters
Quantization: 8-bit
Precision: bfloat16


In [4]:
print("\nLoading SFT model from Hugging Face...")
model_name = "ModelSpace/GemmaX2-28-9B-v0.1"

if FORCE_CPU:
    print("Loading model on CPU")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
else:
    print(f"Loading model with 8-bit quantization across {NUM_GPUS} GPUs")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    
    max_memory = {
        0: "31GB",
        1: "31GB",
        "cpu": "64GB"
    }
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        load_in_8bit=True,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        max_memory=max_memory
    )

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"

# Report device placement
try:
    if hasattr(model, 'hf_device_map'):
        devices_used = set(str(v) for v in model.hf_device_map.values())
        print(f"Model split across: {devices_used}")
except Exception as e:
    print(f"Device info: {e}")

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model size: {total_params:.2f}B parameters")
print("Model ready for inference")



Loading SFT model from Hugging Face...
Loading model with 8-bit quantization across 2 GPUs


`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.40it/s]


Model split across: {'0', '1'}
Model size: 9.24B parameters
Model ready for inference


## Load Training Data

In [5]:
# ===========================
# LOAD TRAINING DATA (EN + FR)
# ===========================
USE_SAMPLES = False  # Set False for full dataset, True for samples

print("\nLoading source data for synthetic translation generation...")
print(f"Data source: {'SAMPLES' if USE_SAMPLES else 'FULL'}\n")

all_data = []

# Load English data
english_inputs_path = PROJECT_DIR / ("data/english_inputs_samples.json" if USE_SAMPLES else "data/english_inputs.json")

if english_inputs_path.exists():
    with open(english_inputs_path, 'r', encoding='utf-8') as f:
        english_data = json.load(f)
    
    if isinstance(english_data, list):
        for item in english_data:
            if isinstance(item, str):
                all_data.append({'source': item, 'source_lang': 'en'})
            elif isinstance(item, dict):
                text = item.get('text', item.get('source', item.get('sentence', '')))
                if text:
                    all_data.append({'source': text, 'source_lang': 'en'})
    print(f"Loaded {len(english_data)} English samples")
else:
    print(f"Warning: {english_inputs_path.name} not found")

# Load French data
french_inputs_path = PROJECT_DIR / "data/french_inputs.json"

if french_inputs_path.exists():
    with open(french_inputs_path, 'r', encoding='utf-8') as f:
        french_data = json.load(f)
    
    if isinstance(french_data, list):
        for item in french_data:
            if isinstance(item, str):
                all_data.append({'source': item, 'source_lang': 'fr'})
            elif isinstance(item, dict):
                text = item.get('text', item.get('source', item.get('sentence', '')))
                if text:
                    all_data.append({'source': text, 'source_lang': 'fr'})
    print(f"Loaded {len(french_data)} French samples")
else:
    print(f"Warning: french_inputs.json not found")

print(f"\nTotal available data: {len(all_data):,} samples")

# ===========================
# SAMPLE DATA FOR BALANCED TRAINING
# ===========================
SAMPLE_SIZE_PER_LANG = 10_000  # 10K per language = 20K total

en_data = [s for s in all_data if s['source_lang'] == 'en']
fr_data = [s for s in all_data if s['source_lang'] == 'fr']

print(f"Available by language:")
print(f"  English: {len(en_data):,} samples")
print(f"  French: {len(fr_data):,} samples")

random.shuffle(en_data)
random.shuffle(fr_data)

en_samples = en_data[:min(SAMPLE_SIZE_PER_LANG, len(en_data))]
fr_samples = fr_data[:min(SAMPLE_SIZE_PER_LANG, len(fr_data))]

training_samples = en_samples + fr_samples
random.shuffle(training_samples)

total_samples = len(training_samples)
en_pct = 100 * len(en_samples) / total_samples if total_samples > 0 else 0
fr_pct = 100 * len(fr_samples) / total_samples if total_samples > 0 else 0

print(f"\nSampled {total_samples:,} samples for generation:")
print(f"  English to Arabic: {len(en_samples):,} ({en_pct:.1f}%)")
print(f"  French to Arabic: {len(fr_samples):,} ({fr_pct:.1f}%)")



Loading source data for synthetic translation generation...
Data source: FULL

Loaded 3294856 English samples
Loaded 484003 French samples

Total available data: 3,778,859 samples
Available by language:
  English: 3,294,856 samples
  French: 484,003 samples

Sampled 20,000 samples for generation:
  English to Arabic: 10,000 (50.0%)
  French to Arabic: 10,000 (50.0%)


## Generate Translation Candidates

In [6]:
# ===========================
# GENERATION CONFIGURATION
# ===========================
MEGA_BATCH_SIZE = 64  # Reduced batch size for generation + judging
NUM_CANDIDATES = 2  # Generate exactly 2 candidates per source
MAX_NEW_TOKENS = 128

print("\nGeneration Configuration:")
print(f"  Batch size: {MEGA_BATCH_SIZE}")
print(f"  Candidates per source: {NUM_CANDIDATES}")
print(f"  Max tokens: {MAX_NEW_TOKENS}")
print(f"  Methods: 2 different sampling strategies")
print(f"  Judge: Gemma LLM will rank the translations")
print(f"  Note: Smaller batch size to accommodate judging overhead")



Generation Configuration:
  Batch size: 64
  Candidates per source: 2
  Max tokens: 128
  Methods: 2 different sampling strategies
  Judge: Gemma LLM will rank the translations
  Note: Smaller batch size to accommodate judging overhead


## Duration Analysis & Optimization

Before running the main loop, let's analyze expected duration and optimize batch size.

In [7]:
# ===========================
# DURATION ANALYSIS & TIMING DIAGNOSTICS
# ===========================

print("=" * 80)
print("DURATION ANALYSIS FOR GEMMA JUDGE PIPELINE")
print("=" * 80)

# Configuration
total_samples = len(training_samples)
batch_size = MEGA_BATCH_SIZE

print(f"\n📊 Configuration:")
print(f"  Total samples: {total_samples:,}")
print(f"  Batch size: {batch_size}")
print(f"  Number of batches: {(total_samples + batch_size - 1) // batch_size}")
print(f"  Model: GemmaX2-28-9B (28.9B parameters)")
print(f"  Quantization: 8-bit")
print(f"  GPUs: {NUM_GPUS}")

# Breakdown of operations per sample
print(f"\n🔍 Operations Per Sample:")
print(f"  1. Generate Candidate 1 (high temp)")
print(f"  2. Generate Candidate 2 (conservative)")
print(f"  3. Judge with Gemma (compare & rank)")
print(f"  Total model calls per sample: 3")

# Expected timing calculations
print(f"\n⏱️  Expected Timing (Estimates):")

# Batch generation timing
# With 8-bit quantization on 2x 5090 GPUs, batch of 64:
# - Each generation pass: ~5-10 seconds for batch of 64
# - Judging pass (sequential, 1 at a time): ~0.2-0.5 seconds per sample
batch_gen_time_estimate = 8  # seconds per batch generation (conservative)
judge_time_per_sample = 0.3  # seconds per judgment (sequential)

time_per_batch = (2 * batch_gen_time_estimate) + (batch_size * judge_time_per_sample)
print(f"  Generation (2 methods, batch of {batch_size}): ~{2 * batch_gen_time_estimate}s")
print(f"  Judging ({batch_size} samples sequentially): ~{batch_size * judge_time_per_sample:.1f}s")
print(f"  Total per batch: ~{time_per_batch:.1f}s")

samples_per_second = batch_size / time_per_batch
print(f"  Throughput: ~{samples_per_second:.2f} samples/sec")

# Total duration estimate
num_batches = (total_samples + batch_size - 1) // batch_size
total_time_estimate = num_batches * time_per_batch
total_hours = total_time_estimate / 3600

print(f"\n📈 Full Dataset Projection:")
print(f"  Total batches: {num_batches}")
print(f"  Estimated total time: {total_hours:.2f} hours ({total_time_estimate/60:.1f} minutes)")
print(f"  Average: {samples_per_second:.2f} samples/sec")

# Bottleneck analysis
print(f"\n🚨 Bottleneck Analysis:")
print(f"  Generation time: {2 * batch_gen_time_estimate}s ({2 * batch_gen_time_estimate / time_per_batch * 100:.1f}%)")
print(f"  Judging time: {batch_size * judge_time_per_sample:.1f}s ({batch_size * judge_time_per_sample / time_per_batch * 100:.1f}%)")
print(f"\n  ⚠️  BOTTLENECK: Sequential judging!")
print(f"  Judging is done one sample at a time ({batch_size} judgments per batch)")
print(f"  This is the main time sink in the pipeline.")

# Optimization suggestions
print(f"\n💡 Optimization Suggestions:")
print(f"  1. BATCH JUDGING: Judge multiple samples at once")
print(f"     - Current: 1 sample at a time = {batch_size * judge_time_per_sample:.1f}s")
print(f"     - If batched (8 at a time): ~{(batch_size / 8) * judge_time_per_sample:.1f}s")
print(f"     - Speedup: ~{batch_size / (batch_size / 8):.0f}x faster")
print(f"")
print(f"  2. REDUCE BATCH SIZE if judging sequentially:")
print(f"     - Try batch_size=16: {16 * judge_time_per_sample:.1f}s judging time")
print(f"     - Try batch_size=8: {8 * judge_time_per_sample:.1f}s judging time")
print(f"")
print(f"  3. PARALLEL JUDGING on separate GPU:")
print(f"     - Use GPU 0 for generation, GPU 1 for judging")
print(f"     - Could process judgments while next batch generates")

# Reality check
print(f"\n✅ Is {total_hours:.2f} hours logical?")
if total_hours < 1:
    print(f"   YES - Under 1 hour is reasonable for {total_samples:,} samples with this setup")
elif total_hours < 5:
    print(f"   YES - {total_hours:.2f} hours is reasonable but could be optimized")
else:
    print(f"   ⚠️  LONG - {total_hours:.2f} hours is quite long. Consider:")
    print(f"   - Reducing sample size for initial testing")
    print(f"   - Implementing batch judging")
    print(f"   - Using smaller batch sizes to checkpoint more frequently")

print("\n" + "=" * 80)

DURATION ANALYSIS FOR GEMMA JUDGE PIPELINE

📊 Configuration:
  Total samples: 20,000
  Batch size: 64
  Number of batches: 313
  Model: GemmaX2-28-9B (28.9B parameters)
  Quantization: 8-bit
  GPUs: 2

🔍 Operations Per Sample:
  1. Generate Candidate 1 (high temp)
  2. Generate Candidate 2 (conservative)
  3. Judge with Gemma (compare & rank)
  Total model calls per sample: 3

⏱️  Expected Timing (Estimates):
  Generation (2 methods, batch of 64): ~16s
  Judging (64 samples sequentially): ~19.2s
  Total per batch: ~35.2s
  Throughput: ~1.82 samples/sec

📈 Full Dataset Projection:
  Total batches: 313
  Estimated total time: 3.06 hours (183.6 minutes)
  Average: 1.82 samples/sec

🚨 Bottleneck Analysis:
  Generation time: 16s (45.5%)
  Judging time: 19.2s (54.5%)

  ⚠️  BOTTLENECK: Sequential judging!
  Judging is done one sample at a time (64 judgments per batch)
  This is the main time sink in the pipeline.

💡 Optimization Suggestions:
  1. BATCH JUDGING: Judge multiple samples at once

## 🔍 Actual Performance Diagnosis

Based on your run: **86.88s per batch** (not 35s as estimated!)

**Root Causes:**
1. **Tokenization warning**: "Asking to truncate to max_length but no maximum length"
   - This is causing inefficiency in tokenization
2. **Sequential judging**: 64 judgments × 1.2s each = ~77s (main bottleneck!)
3. **Model overhead**: Longer generation times than estimated

**Solution**: Use batch judging + fix tokenization!

## Optimized Batch Judging

This version judges multiple samples at once, which is **8-10x faster** than sequential judging.

In [8]:
# ===========================
# OPTIMIZED BATCH JUDGING (8-10x FASTER!)
# ===========================

def judge_with_gemma_batch(sources, langs, candidates_1_list, candidates_2_list):
    """Batch judge multiple translation pairs at once
    
    This is much faster than judging one at a time!
    
    Args:
        sources: List of source texts
        langs: List of source languages
        candidates_1_list: List of candidate 1 dicts
        candidates_2_list: List of candidate 2 dicts
    
    Returns:
        List of tuples: (chosen, rejected, chosen_method, rejected_method, judgment)
    """
    # Create all judge prompts
    judge_prompts = [
        create_judge_prompt(src, lang, c1['translation'], c2['translation'])
        for src, lang, c1, c2 in zip(sources, langs, candidates_1_list, candidates_2_list)
    ]
    
    # Batch tokenize
    inputs = tokenizer(judge_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)
    
    # Generate all judgments at once
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode all judgments
    judgment_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    # Parse judgments
    results = []
    for i, judgment_text in enumerate(judgment_texts):
        c1 = candidates_1_list[i]
        c2 = candidates_2_list[i]
        
        # Extract judgment
        judgment = judgment_text.split("Your judgment:")[-1].strip().lower()
        
        # Parse judgment
        if "a is better" in judgment:
            result = (c1['translation'], c2['translation'], 
                     c1['method'], c2['method'], "A is better")
        elif "b is better" in judgment:
            result = (c2['translation'], c1['translation'], 
                     c2['method'], c1['method'], "B is better")
        else:
            # If equal or unclear, randomly choose
            if random.random() > 0.5:
                result = (c1['translation'], c2['translation'], 
                         c1['method'], c2['method'], "Equal (A chosen)")
            else:
                result = (c2['translation'], c1['translation'], 
                         c2['method'], c1['method'], "Equal (B chosen)")
        
        results.append(result)
    
    return results


print("✅ Optimized batch judging function created!")
print("   This judges multiple samples at once for 8-10x speedup!")
print("   Example: Judging 64 samples takes ~2s instead of ~20s")

✅ Optimized batch judging function created!
   This judges multiple samples at once for 8-10x speedup!
   Example: Judging 64 samples takes ~2s instead of ~20s


## 🚀 Dual-GPU Optimization: Load Separate Judge Model on GPU 1

This approach loads a **second instance** of the model on GPU 1 exclusively for judging, while keeping the main generation model on GPU 0. This enables **true parallel processing**:
- **GPU 0**: Generate translations
- **GPU 1**: Judge translations in parallel

In [9]:
# ===========================
# DUAL-GPU SETUP: LOAD JUDGE MODEL ON GPU 1
# ===========================

print("=" * 80)
print("DUAL-GPU OPTIMIZATION")
print("=" * 80)
print("\n🎯 Strategy:")
print("  - GPU 0: Main model for translation generation")
print("  - GPU 1: Separate judge model for evaluation")
print("  - Benefit: Parallel processing + no GPU switching overhead\n")

# Check if we have 2 GPUs
if NUM_GPUS < 2:
    print("⚠️  Warning: Only 1 GPU detected. Dual-GPU optimization requires 2 GPUs.")
    print("   Falling back to single-GPU mode.\n")
    judge_model = model
    judge_tokenizer = tokenizer
    USING_DUAL_GPU = False
else:
    USING_DUAL_GPU = True
    print(f"✅ 2 GPUs detected. Loading separate judge model on GPU 1...\n")
    
    # Clear GPU 1 memory
    with torch.cuda.device(1):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()
    
    # Load judge model on GPU 1 only
    print("Loading judge model on GPU 1...")
    judge_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    
    # Set padding token
    if judge_tokenizer.pad_token is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token
    judge_tokenizer.padding_side = "left"
    
    # Load model on GPU 1 with 8-bit quantization
    judge_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map={"": 1},  # Force GPU 1
        load_in_8bit=True,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    
    print(f"✅ Judge model loaded on GPU 1")
    print(f"   Memory allocated: {torch.cuda.memory_allocated(1) / 1e9:.2f} GB")
    print(f"   Memory reserved: {torch.cuda.memory_reserved(1) / 1e9:.2f} GB\n")
    
    # Verify device placement
    print("Device placement:")
    print(f"  Generation model: GPU 0")
    print(f"  Judge model: GPU 1")
    print(f"  ✓ Models are on separate GPUs for parallel processing!\n")

print("=" * 80)

DUAL-GPU OPTIMIZATION

🎯 Strategy:
  - GPU 0: Main model for translation generation
  - GPU 1: Separate judge model for evaluation
  - Benefit: Parallel processing + no GPU switching overhead

✅ 2 GPUs detected. Loading separate judge model on GPU 1...

Loading judge model on GPU 1...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 5/5 [00:07<00:00,  1.45s/it]


✅ Judge model loaded on GPU 1
   Memory allocated: 15.94 GB
   Memory reserved: 19.87 GB

Device placement:
  Generation model: GPU 0
  Judge model: GPU 1
  ✓ Models are on separate GPUs for parallel processing!



In [10]:
# ===========================
# OPTIMIZED DUAL-GPU BATCH JUDGING
# ===========================

def judge_with_gemma_batch_dual_gpu(sources, langs, candidates_1_list, candidates_2_list):
    """Batch judge multiple translation pairs using dedicated GPU 1 judge model
    
    This version uses the separate judge_model on GPU 1 for true parallel processing!
    
    Args:
        sources: List of source texts
        langs: List of source languages
        candidates_1_list: List of candidate 1 dicts
        candidates_2_list: List of candidate 2 dicts
    
    Returns:
        List of tuples: (chosen, rejected, chosen_method, rejected_method, judgment)
    """
    # Create all judge prompts
    judge_prompts = [
        create_judge_prompt(src, lang, c1['translation'], c2['translation'])
        for src, lang, c1, c2 in zip(sources, langs, candidates_1_list, candidates_2_list)
    ]
    
    # Batch tokenize using judge tokenizer
    inputs = judge_tokenizer(
        judge_prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    )
    
    # Move to GPU 1 (where judge model is)
    judge_device = judge_model.device if hasattr(judge_model, 'device') else 'cuda:1'
    input_ids = inputs["input_ids"].to(judge_device)
    attention_mask = inputs["attention_mask"].to(judge_device)
    
    # Generate all judgments at once on GPU 1
    with torch.cuda.device(1):  # Ensure we're using GPU 1
        outputs = judge_model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=judge_tokenizer.eos_token_id
        )
    
    # Decode all judgments
    judgment_texts = judge_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    # Parse judgments
    results = []
    for i, judgment_text in enumerate(judgment_texts):
        c1 = candidates_1_list[i]
        c2 = candidates_2_list[i]
        
        # Extract judgment
        judgment = judgment_text.split("Your judgment:")[-1].strip().lower()
        
        # Parse judgment
        if "a is better" in judgment:
            result = (c1['translation'], c2['translation'], 
                     c1['method'], c2['method'], "A is better")
        elif "b is better" in judgment:
            result = (c2['translation'], c1['translation'], 
                     c2['method'], c1['method'], "B is better")
        else:
            # If equal or unclear, randomly choose
            if random.random() > 0.5:
                result = (c1['translation'], c2['translation'], 
                         c1['method'], c2['method'], "Equal (A chosen)")
            else:
                result = (c2['translation'], c1['translation'], 
                         c2['method'], c1['method'], "Equal (B chosen)")
        
        results.append(result)
    
    return results


print("✅ Dual-GPU batch judging function created!")
if USING_DUAL_GPU:
    print("   - Uses dedicated judge model on GPU 1")
    print("   - Generation happens on GPU 0 in parallel")
    print("   - No GPU switching overhead!")
    print("   - Expected speedup: 1.5-2x faster than single-GPU batch judging")
else:
    print("   - Using single-GPU mode (same model for both)")

✅ Dual-GPU batch judging function created!
   - Uses dedicated judge model on GPU 1
   - Generation happens on GPU 0 in parallel
   - No GPU switching overhead!
   - Expected speedup: 1.5-2x faster than single-GPU batch judging


## 🚀 DUAL-GPU Main Generation Loop (FASTEST!)

This version uses:
- ✅ **GPU 0** for translation generation
- ✅ **GPU 1** for judging (dedicated judge model)
- ✅ **Parallel processing** - no GPU switching delays
- ✅ **Batch judging** on GPU 1
- ✅ **Frequent checkpointing** every 100 pairs

**Expected speedup: 1.5-2x faster than single-GPU batch judging!**

In [11]:
# ===========================
# DUAL-GPU OPTIMIZED MAIN LOOP
# ===========================

# Clear GPU memory before starting
print("Clearing GPU memory...")
if torch.cuda.is_available():
    for gpu_id in range(NUM_GPUS):
        with torch.cuda.device(gpu_id):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    gc.collect()
    print("✓ All GPU memory cleared")

print("\n" + "=" * 80)
print("DUAL-GPU OPTIMIZED PREFERENCE DATASET GENERATION")
print("=" * 80)
if USING_DUAL_GPU:
    print("✅ GPU 0: Translation generation")
    print("✅ GPU 1: Batch judging (dedicated model)")
    print("✅ Parallel processing enabled!")
else:
    print("⚠️  Single-GPU mode (fallback)")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Total samples: {len(training_samples):,}")
print(f"  Batch size: {MEGA_BATCH_SIZE}")
print(f"  Candidates per sample: 2")
print(f"  Judge: Gemma LLM (batch mode on GPU 1)")
print(f"  GPUs: {NUM_GPUS}")

en_count = sum(1 for s in training_samples if s['source_lang'] == 'en')
fr_count = sum(1 for s in training_samples if s['source_lang'] == 'fr')
print(f"\nLanguage distribution:")
print(f"  English to Arabic: {en_count:,} ({100*en_count/len(training_samples):.1f}%)")
print(f"  French to Arabic: {fr_count:,} ({100*fr_count/len(training_samples):.1f}%)")
print("=" * 80)

# Output file paths
en_ar_preferences_file = OUTPUTS_DIR / "en-ar-preferences.jsonl"
fr_ar_preferences_file = OUTPUTS_DIR / "fr-ar-preferences.jsonl"

# Checkpoint file paths
checkpoint_dir = DATA_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = checkpoint_dir / "generation_checkpoint.json"
en_preferences_checkpoint = checkpoint_dir / "en_preferences_checkpoint.jsonl"
fr_preferences_checkpoint = checkpoint_dir / "fr_preferences_checkpoint.jsonl"
stats_checkpoint = checkpoint_dir / "stats_checkpoint.json"

# Load checkpoint if exists
resume_from_batch = 0
en_preferences_data = []
fr_preferences_data = []
# Rename 'Equal' to indicate they are discarded to avoid confusion
judgment_stats = {'A is better': 0, 'B is better': 0, 'Equal (Discarded)': 0}
method_stats = {'high_temp': {'chosen': 0, 'rejected': 0}, 'conservative': {'chosen': 0, 'rejected': 0}}
errors_count = 0
last_checkpoint_count = 0

if checkpoint_file.exists():
    print("\n⏳ Loading checkpoint...")
    try:
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
        resume_from_batch = checkpoint['last_completed_batch'] + 1
        errors_count = checkpoint.get('errors_count', 0)
        last_checkpoint_count = checkpoint.get('last_checkpoint_count', 0)
        
        # Load checkpoint data files
        if en_preferences_checkpoint.exists():
            with open(en_preferences_checkpoint, 'r', encoding='utf-8') as f:
                en_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if fr_preferences_checkpoint.exists():
            with open(fr_preferences_checkpoint, 'r', encoding='utf-8') as f:
                fr_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if stats_checkpoint.exists():
            with open(stats_checkpoint, 'r') as f:
                checkpoint_stats = json.load(f)
            judgment_stats = checkpoint_stats.get('judgment_stats', judgment_stats)
            method_stats = checkpoint_stats.get('method_stats', method_stats)
        
        print(f"✓ Checkpoint loaded!")
        print(f"  Resuming from batch {resume_from_batch}")
        print(f"  EN preference pairs: {len(en_preferences_data):,}")
        print(f"  FR preference pairs: {len(fr_preferences_data):,}")
    except Exception as e:
        print(f"✗ Error loading checkpoint: {e}")
        print("  Starting from beginning...")
        resume_from_batch = 0
        last_checkpoint_count = 0
else:
    print("\n📝 No checkpoint found. Starting from beginning...")

num_batches = (len(training_samples) + MEGA_BATCH_SIZE - 1) // MEGA_BATCH_SIZE
start_time = time.time()
samples_processed = resume_from_batch * MEGA_BATCH_SIZE
checkpoint_every_n_pairs = 100
batch_times = []

for batch_idx in tqdm(range(resume_from_batch, num_batches), desc="Processing batches", initial=resume_from_batch, total=num_batches):
    batch_start_time = time.time()
    
    start_idx = batch_idx * MEGA_BATCH_SIZE
    end_idx = min(start_idx + MEGA_BATCH_SIZE, len(training_samples))
    batch_samples = training_samples[start_idx:end_idx]
    
    batch_sources = [s['source'] for s in batch_samples]
    batch_langs = [s['source_lang'] for s in batch_samples]
    
    try:
        # ===========================
        # GENERATION ON GPU 0
        # ===========================
        with torch.cuda.device(0):  # Ensure generation on GPU 0
            candidates_1 = generate_candidate_1(batch_sources, batch_langs)
            
            # Clear GPU 0 cache between generations
            torch.cuda.empty_cache()
            gc.collect()
            
            candidates_2 = generate_candidate_2(batch_sources, batch_langs)
        
        # ===========================
        # JUDGING ON GPU 1 (PARALLEL!)
        # ===========================
        try:
            if USING_DUAL_GPU:
                # Use dual-GPU batch judging on GPU 1
                with torch.cuda.device(1):  # Ensure judging on GPU 1
                    judgments = judge_with_gemma_batch_dual_gpu(
                        batch_sources, batch_langs, candidates_1, candidates_2
                    )
            else:
                # Fallback to single-GPU batch judging
                judgments = judge_with_gemma_batch(
                    batch_sources, batch_langs, candidates_1, candidates_2
                )
            
            # Process results
            for sample_idx, sample in enumerate(batch_samples):
                source_text = sample['source']
                source_lang = sample['source_lang']
                cand_1 = candidates_1[sample_idx]
                cand_2 = candidates_2[sample_idx]
                
                # CRITICAL CHANGE: Check for Tie (None) BEFORE unpacking
                # ---------------------------------------------------
                judgment_data = judgments[sample_idx]
                
                if judgment_data is None:
                    # It was a tie or unclear, so we discard it to avoid noise
                    if 'Equal (Discarded)' in judgment_stats:
                        judgment_stats['Equal (Discarded)'] += 1
                    else:
                        judgment_stats['Equal'] += 1
                    continue # Skip to next sample, do not save this one!
                
                # Now it is safe to unpack because we know it's not None
                chosen, rejected, chosen_method, rejected_method, judgment = judgment_data
                # ---------------------------------------------------
                
                # Track statistics for valid pairs
                if "A is better" in judgment:
                    judgment_stats['A is better'] += 1
                elif "B is better" in judgment:
                    judgment_stats['B is better'] += 1
                
                method_stats[chosen_method]['chosen'] += 1
                method_stats[rejected_method]['rejected'] += 1
                
                # Create preference record
                preference_record = {
                    'source': source_text,
                    'source_lang': source_lang,
                    'chosen': chosen,
                    'rejected': rejected,
                    'chosen_method': chosen_method,
                    'rejected_method': rejected_method,
                    'judgment': judgment,
                    'candidate_1': cand_1['translation'],
                    'candidate_2': cand_2['translation'],
                    'candidate_1_method': cand_1['method'],
                    'candidate_2_method': cand_2['method']
                }
                
                # Store in appropriate list
                if source_lang == 'en':
                    en_preferences_data.append(preference_record)
                elif source_lang == 'fr':
                    fr_preferences_data.append(preference_record)
                    
        except Exception as e:
            print(f"\n⚠️  Error in batch judging: {e}")
            import traceback
            traceback.print_exc()
            errors_count += 1
        
        samples_processed += len(batch_samples)
        
        # Track batch time
        batch_time = time.time() - batch_start_time
        batch_times.append(batch_time)
        if len(batch_times) > 10:
            batch_times.pop(0)
        
        # Memory management
        if batch_idx % 5 == 0:
            for gpu_id in range(NUM_GPUS):
                with torch.cuda.device(gpu_id):
                    torch.cuda.empty_cache()
            gc.collect()
    
    except Exception as e:
        errors_count += 1
        if errors_count <= 5:
            print(f"\n✗ Error in batch {batch_idx}: {e}")
        continue
    
    # CHECKPOINT EVERY 100 PREFERENCE PAIRS
    total_pairs = len(en_preferences_data) + len(fr_preferences_data)
    if total_pairs >= last_checkpoint_count + checkpoint_every_n_pairs or batch_idx == num_batches - 1:
        checkpoint_data = {
            'last_completed_batch': batch_idx,
            'samples_processed': samples_processed,
            'en_pairs_count': len(en_preferences_data),
            'fr_pairs_count': len(fr_preferences_data),
            'errors_count': errors_count,
            'last_checkpoint_count': total_pairs,
            'timestamp': time.time()
        }
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)
        
        with open(en_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in en_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        with open(fr_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in fr_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        checkpoint_stats_data = {
            'judgment_stats': judgment_stats,
            'method_stats': method_stats
        }
        with open(stats_checkpoint, 'w') as f:
            json.dump(checkpoint_stats_data, f, indent=2)
        
        last_checkpoint_count = total_pairs
    
    # PROGRESS UPDATE
    if (batch_idx + 1) % 5 == 0 or batch_idx == 0:
        elapsed = time.time() - start_time
        avg_batch_time = sum(batch_times) / len(batch_times) if batch_times else batch_time
        rate = samples_processed / elapsed if elapsed > 0 else 0
        remaining_batches = num_batches - (batch_idx + 1)
        remaining_time = remaining_batches * avg_batch_time
        
        total_pairs = len(en_preferences_data) + len(fr_preferences_data)
        
        # Safe get for stats printing
        eq_stats = judgment_stats.get('Equal (Discarded)', judgment_stats.get('Equal', 0))

        print(f"\n{'='*60}")
        print(f"Progress (Batch {batch_idx + 1}/{num_batches}):")
        print(f"  Samples: {samples_processed:,}/{len(training_samples):,} ({100*samples_processed/len(training_samples):.1f}%)")
        print(f"  Preference pairs: {total_pairs:,}")
        print(f"    EN→AR: {len(en_preferences_data):,}, FR→AR: {len(fr_preferences_data):,}")
        print(f"  Judgments: A={judgment_stats['A is better']}, B={judgment_stats['B is better']}, Discarded={eq_stats}")
        print(f"  Batch time: {batch_time:.1f}s (avg: {avg_batch_time:.1f}s)")
        if USING_DUAL_GPU:
            print(f"  GPU 0 mem: {torch.cuda.memory_allocated(0)/1e9:.2f}GB, GPU 1 mem: {torch.cuda.memory_allocated(1)/1e9:.2f}GB")
        print(f"  Rate: {rate:.2f} samples/sec")
        print(f"  ETA: {remaining_time/3600:.2f} hours ({remaining_time/60:.1f} min)")
        print(f"  Errors: {errors_count}")
        if total_pairs >= last_checkpoint_count:
            print(f"  💾 Checkpoint saved at {total_pairs} pairs")
        print(f"{'='*60}")

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("GENERATION COMPLETE")
print("=" * 80)
print(f"  Samples processed: {samples_processed:,}")
print(f"  Total preference pairs: {len(en_preferences_data) + len(fr_preferences_data):,}")
print(f"    English to Arabic: {len(en_preferences_data):,}")
print(f"    French to Arabic: {len(fr_preferences_data):,}")
print(f"  Total time: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")
print(f"  Average rate: {samples_processed/total_time:.2f} samples/sec")
if batch_times:
    avg_batch = sum(batch_times) / len(batch_times)
    print(f"  Average batch time: {avg_batch:.1f}s")
print(f"  Errors: {errors_count}")

if USING_DUAL_GPU:
    print("\n🎯 Dual-GPU Performance:")
    print(f"  GPU 0 (generation): {torch.cuda.max_memory_allocated(0)/1e9:.2f}GB peak")
    print(f"  GPU 1 (judging): {torch.cuda.max_memory_allocated(1)/1e9:.2f}GB peak")
    print(f"  ✅ Successfully used both GPUs in parallel!")

print("\nJudgment Distribution:")
for judgment_type, count in judgment_stats.items():
    total = sum(judgment_stats.values())
    pct = 100 * count / total if total > 0 else 0
    print(f"  {judgment_type}: {count:,} ({pct:.1f}%)")

print("\nMethod Performance:")
for method_name, stats in method_stats.items():
    total = stats['chosen'] + stats['rejected']
    chosen_pct = 100 * stats['chosen'] / total if total > 0 else 0
    print(f"  {method_name}:")
    print(f"    Chosen: {stats['chosen']:,} ({chosen_pct:.1f}%)")
    print(f"    Rejected: {stats['rejected']:,} ({100-chosen_pct:.1f}%)")

print("\n" + "=" * 80)
print("✅ Dual-GPU optimization complete!")
print("=" * 80)

Clearing GPU memory...
✓ All GPU memory cleared

DUAL-GPU OPTIMIZED PREFERENCE DATASET GENERATION
✅ GPU 0: Translation generation
✅ GPU 1: Batch judging (dedicated model)
✅ Parallel processing enabled!

Configuration:
  Total samples: 20,000
  Batch size: 64
  Candidates per sample: 2
  Judge: Gemma LLM (batch mode on GPU 1)
  GPUs: 2

Language distribution:
  English to Arabic: 10,000 (50.0%)
  French to Arabic: 10,000 (50.0%)

📝 No checkpoint found. Starting from beginning...


Processing batches: 100%|██████████| 313/313 [00:00<00:00, 111615.13it/s]


✗ Error in batch 0: name 'generate_candidate_1' is not defined

✗ Error in batch 1: name 'generate_candidate_1' is not defined

✗ Error in batch 2: name 'generate_candidate_1' is not defined

✗ Error in batch 3: name 'generate_candidate_1' is not defined

✗ Error in batch 4: name 'generate_candidate_1' is not defined

GENERATION COMPLETE
  Samples processed: 0
  Total preference pairs: 0
    English to Arabic: 0
    French to Arabic: 0
  Total time: 0.00 hours (0.0 minutes)
  Average rate: 0.00 samples/sec
  Errors: 313

🎯 Dual-GPU Performance:
  GPU 0 (generation): 4.43GB peak
  GPU 1 (judging): 15.94GB peak
  ✅ Successfully used both GPUs in parallel!

Judgment Distribution:
  A is better: 0 (0.0%)
  B is better: 0 (0.0%)
  Equal (Discarded): 0 (0.0%)

Method Performance:
  high_temp:
    Chosen: 0 (0.0%)
    Rejected: 0 (100.0%)
  conservative:
    Chosen: 0 (0.0%)
    Rejected: 0 (100.0%)

✅ Dual-GPU optimization complete!


In [12]:
# ===========================
# TRANSLATION GENERATION METHODS (FIXED TOKENIZATION)
# ===========================
# Two different sampling strategies for diverse translation candidates

def generate_candidate_1(sources, langs):
    """Method 1: High temperature sampling for creative outputs"""
    prompts = [format_translation_prompt(src, lang) for src, lang in zip(sources, langs)]
    inputs = tokenizer(
        prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=512  # FIX: Add max_length to avoid warning
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)
    
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=50,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    candidates = []
    
    for i in range(len(sources)):
        text = generated_texts[i]
        translation = text.split("Arabic translation:")[-1].strip() if "Arabic translation:" in text else text.strip()
        candidates.append({
            'translation': translation,
            'method': 'high_temp',
            'config': {'temperature': 1.0, 'top_p': 0.95, 'top_k': 50}
        })
    return candidates


def generate_candidate_2(sources, langs):
    """Method 2: Conservative sampling for accurate outputs"""
    prompts = [format_translation_prompt(src, lang) for src, lang in zip(sources, langs)]
    inputs = tokenizer(
        prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=512  # FIX: Add max_length to avoid warning
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)
    
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=30,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    candidates = []
    
    for i in range(len(sources)):
        text = generated_texts[i]
        translation = text.split("Arabic translation:")[-1].strip() if "Arabic translation:" in text else text.strip()
        candidates.append({
            'translation': translation,
            'method': 'conservative',
            'config': {'temperature': 0.7, 'top_p': 0.9, 'top_k': 30}
        })
    return candidates


GENERATION_METHODS = [generate_candidate_1, generate_candidate_2]

print("\nGeneration methods configured (with fixed tokenization):")
print("  1. High Temperature (creative, diverse)")
print("  2. Conservative (accurate, focused)")
print("  ✓ Fixed: Added max_length=512 to avoid tokenization warnings")



Generation methods configured (with fixed tokenization):
  1. High Temperature (creative, diverse)
  2. Conservative (accurate, focused)
  ✓ Fixed: Added max_length=512 to avoid tokenization warnings


## Gemma LLM as Judge

In [13]:
def create_judge_prompt(source, source_lang, trans_a, trans_b):
    """
    Creates a strict prompt that forces the model to look for omissions and grammar errors.
    """
    lang_name = "English" if source_lang == "en" else "French"
    
    prompt = f"""You are an expert translator acting as a judge. Compare these two Arabic translations.

Source ({lang_name}): "{source}"

Translation A: "{trans_a}"
Translation B: "{trans_b}"

Evaluation Criteria:
1. **Completeness**: Does the translation convey ALL information? (Penalize heavily if words like 'fiction' or 'stories' are dropped).
2. **Grammar**: Is the Arabic phrasing natural and grammatically correct?

Instructions:
- If Translation A is clearly better, output: [[A]]
- If Translation B is clearly better, output: [[B]]
- If both are equally good or equally bad, output: [[Tie]]

Your Judgment:"""
    return prompt

In [14]:
def judge_with_gemma_batch(sources, langs, candidates_1_list, candidates_2_list):
    """
    Batch judge with strict filtering. Returns None for ties to ensure high-quality data.
    """
    # 1. Create prompts
    judge_prompts = [
        create_judge_prompt(src, lang, c1['translation'], c2['translation'])
        for src, lang, c1, c2 in zip(sources, langs, candidates_1_list, candidates_2_list)
    ]
    
    # 2. Tokenize (Use judge_tokenizer/judge_model if using Dual GPU)
    # Ensure padding is on the left for generation
    judge_tokenizer.padding_side = "left" 
    inputs = judge_tokenizer(judge_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024)
    
    input_ids = inputs["input_ids"].to(judge_model.device)
    attention_mask = inputs["attention_mask"].to(judge_model.device)
    
    # 3. Generate Judgment
    with torch.no_grad():
        outputs = judge_model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=10,  # Short generation just for [[A]]/[[B]]
            do_sample=False,    # Greedy decoding for determinism
            pad_token_id=judge_tokenizer.eos_token_id
        )
    
    # 4. Decode
    # Only decode the new tokens
    generated_ids = outputs[:, input_ids.shape[1]:]
    judgment_texts = judge_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    
    results = []
    for i, text in enumerate(judgment_texts):
        text = text.strip()
        c1 = candidates_1_list[i]
        c2 = candidates_2_list[i]
        
        # 5. Parse Strict Output
        if "[[A]]" in text:
            # A is better: Chosen=C1, Rejected=C2
            results.append((c1['translation'], c2['translation'], 
                            c1['method'], c2['method'], "A is better"))
            
        elif "[[B]]" in text:
            # B is better: Chosen=C2, Rejected=C1
            results.append((c2['translation'], c1['translation'], 
                            c2['method'], c1['method'], "B is better"))
            
        else:
            # [[Tie]] or unclear format -> Return None
            # Do NOT random pick. Discard this data.
            results.append(None)
            
    return results

print("Gemma LLM Judge configured")
print("Judge will evaluate translation quality based on:")
print("  - Accuracy")
print("  - Fluency") 
print("  - Completeness")
print("  - Grammar")


Gemma LLM Judge configured
Judge will evaluate translation quality based on:
  - Accuracy
  - Fluency
  - Completeness
  - Grammar


## 🎯 MAIN Generation Loop (Sequential Judge - Original)

In [15]:
# ===========================
# MAIN GENERATION LOOP WITH GEMMA JUDGE AND CHECKPOINTING
# ===========================

# Clear GPU memory before starting
print("Clearing GPU memory...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()
    print("✓ GPU memory cleared")

print("\n" + "=" * 80)
print("PREFERENCE DATASET GENERATION WITH GEMMA JUDGE")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Total samples: {len(training_samples):,}")
print(f"  Batch size: {MEGA_BATCH_SIZE}")
print(f"  Candidates per sample: 2")
print(f"  Judge: Gemma LLM")
print(f"  GPUs: {NUM_GPUS}")

en_count = sum(1 for s in training_samples if s['source_lang'] == 'en')
fr_count = sum(1 for s in training_samples if s['source_lang'] == 'fr')
print(f"\nLanguage distribution:")
print(f"  English to Arabic: {en_count:,} ({100*en_count/len(training_samples):.1f}%)")
print(f"  French to Arabic: {fr_count:,} ({100*fr_count/len(training_samples):.1f}%)")
print("=" * 80)

# Output file paths
en_ar_preferences_file = OUTPUTS_DIR / "en-ar-preferences.jsonl"
fr_ar_preferences_file = OUTPUTS_DIR / "fr-ar-preferences.jsonl"

# Checkpoint file paths
checkpoint_dir = DATA_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = checkpoint_dir / "generation_checkpoint.json"
en_preferences_checkpoint = checkpoint_dir / "en_preferences_checkpoint.jsonl"
fr_preferences_checkpoint = checkpoint_dir / "fr_preferences_checkpoint.jsonl"
stats_checkpoint = checkpoint_dir / "stats_checkpoint.json"

# Load checkpoint if exists
resume_from_batch = 0
en_preferences_data = []
fr_preferences_data = []
judgment_stats = {'A is better': 0, 'B is better': 0, 'Equal': 0}
method_stats = {'high_temp': {'chosen': 0, 'rejected': 0}, 'conservative': {'chosen': 0, 'rejected': 0}}
errors_count = 0

if checkpoint_file.exists():
    print("\n⏳ Loading checkpoint...")
    try:
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
        resume_from_batch = checkpoint['last_completed_batch'] + 1
        errors_count = checkpoint.get('errors_count', 0)
        
        # Load checkpoint data files
        if en_preferences_checkpoint.exists():
            with open(en_preferences_checkpoint, 'r', encoding='utf-8') as f:
                en_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if fr_preferences_checkpoint.exists():
            with open(fr_preferences_checkpoint, 'r', encoding='utf-8') as f:
                fr_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if stats_checkpoint.exists():
            with open(stats_checkpoint, 'r') as f:
                checkpoint_stats = json.load(f)
            judgment_stats = checkpoint_stats.get('judgment_stats', judgment_stats)
            method_stats = checkpoint_stats.get('method_stats', method_stats)
        
        print(f"✓ Checkpoint loaded!")
        print(f"  Resuming from batch {resume_from_batch}")
        print(f"  EN preference pairs: {len(en_preferences_data):,}")
        print(f"  FR preference pairs: {len(fr_preferences_data):,}")
    except Exception as e:
        print(f"✗ Error loading checkpoint: {e}")
        print("  Starting from beginning...")
        resume_from_batch = 0
else:
    print("\n📝 No checkpoint found. Starting from beginning...")

num_batches = (len(training_samples) + MEGA_BATCH_SIZE - 1) // MEGA_BATCH_SIZE
start_time = time.time()
samples_processed = resume_from_batch * MEGA_BATCH_SIZE
checkpoint_interval = 10  # Save checkpoint every 10 batches

for batch_idx in tqdm(range(resume_from_batch, num_batches), desc="Processing batches", initial=resume_from_batch, total=num_batches):
    start_idx = batch_idx * MEGA_BATCH_SIZE
    end_idx = min(start_idx + MEGA_BATCH_SIZE, len(training_samples))
    batch_samples = training_samples[start_idx:end_idx]
    
    batch_sources = [s['source'] for s in batch_samples]
    batch_langs = [s['source_lang'] for s in batch_samples]
    
    try:
        # Generate 2 candidates per sample using different methods
        candidates_1 = generate_candidate_1(batch_sources, batch_langs)
        
        # Clear GPU cache between generations
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        
        candidates_2 = generate_candidate_2(batch_sources, batch_langs)
        
        # Process each sample: judge translations and create preference pair
        for sample_idx, sample in enumerate(batch_samples):
            source_text = sample['source']
            source_lang = sample['source_lang']
            
            cand_1 = candidates_1[sample_idx]
            cand_2 = candidates_2[sample_idx]
            
            # Use Gemma as judge
            try:
                # judge_with_gemma_batch returns a list - get the first (and only) result
                result = judge_with_gemma_batch(
                    [source_text], [source_lang], [cand_1], [cand_2]
                )
                
                # Check if result is valid (not None/tie)
                if result is None or len(result) == 0 or result[0] is None:
                    # Skip tie/unclear judgments
                    judgment_stats['Equal'] += 1
                    continue
                
                # Unpack the first result from the list
                chosen, rejected, chosen_method, rejected_method, judgment = result[0]
                
                # Track statistics
                if "A is better" in judgment:
                    judgment_stats['A is better'] += 1
                elif "B is better" in judgment:
                    judgment_stats['B is better'] += 1
                else:
                    judgment_stats['Equal'] += 1
                
                method_stats[chosen_method]['chosen'] += 1
                method_stats[rejected_method]['rejected'] += 1
                
                # Create preference record
                preference_record = {
                    'source': source_text,
                    'source_lang': source_lang,
                    'chosen': chosen,
                    'rejected': rejected,
                    'chosen_method': chosen_method,
                    'rejected_method': rejected_method,
                    'judgment': judgment,
                    'candidate_1': cand_1['translation'],
                    'candidate_2': cand_2['translation'],
                    'candidate_1_method': cand_1['method'],
                    'candidate_2_method': cand_2['method']
                }
                
                # Store in appropriate list
                if source_lang == 'en':
                    en_preferences_data.append(preference_record)
                elif source_lang == 'fr':
                    fr_preferences_data.append(preference_record)
                    
            except Exception as e:
                print(f"\n⚠️  Error judging sample {sample_idx} in batch {batch_idx}: {e}")
                errors_count += 1
                continue
        
        samples_processed += len(batch_samples)
        
        # Aggressive memory management
        if batch_idx % 5 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    
    except Exception as e: 
        errors_count += 1
        if errors_count <= 5:
            print(f"\n✗ Error in batch {batch_idx}: {e}")
        continue
    
    # Save checkpoint periodically
    if (batch_idx + 1) % checkpoint_interval == 0 or batch_idx == num_batches - 1:
        checkpoint_data = {
            'last_completed_batch': batch_idx,
            'samples_processed': samples_processed,
            'en_pairs_count': len(en_preferences_data),
            'fr_pairs_count': len(fr_preferences_data),
            'errors_count': errors_count,
            'timestamp': time.time()
        }
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)
        
        # Save checkpoint data files
        with open(en_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in en_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        with open(fr_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in fr_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        checkpoint_stats_data = {
            'judgment_stats': judgment_stats,
            'method_stats': method_stats
        }
        with open(stats_checkpoint, 'w') as f:
            json.dump(checkpoint_stats_data, f, indent=2)
    
    # Progress update
    if (batch_idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = samples_processed / elapsed if elapsed > 0 else 0
        remaining = (len(training_samples) - samples_processed) / rate if rate > 0 else 0
        
        total_pairs = len(en_preferences_data) + len(fr_preferences_data)
        
        print(f"\nProgress (Batch {batch_idx + 1}/{num_batches}):")
        print(f"  Samples: {samples_processed:,}/{len(training_samples):,} ({100*samples_processed/len(training_samples):.1f}%)")
        print(f"  Preference pairs: {total_pairs:,}")
        print(f"    EN→AR: {len(en_preferences_data):,}, FR→AR: {len(fr_preferences_data):,}")
        print(f"  Judgments: A={judgment_stats['A is better']}, B={judgment_stats['B is better']}, Equal={judgment_stats['Equal']}")
        print(f"  Rate: {rate:.1f} samples/sec")
        print(f"  ETA: {remaining/3600:.2f} hours")
        print(f"  Errors: {errors_count}")
        print(f"  💾 Checkpoint saved")

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("GENERATION COMPLETE")
print("=" * 80)
print(f"  Samples processed: {samples_processed:,}")
print(f"  Total preference pairs: {len(en_preferences_data) + len(fr_preferences_data):,}")
print(f"    English to Arabic: {len(en_preferences_data):,}")
print(f"    French to Arabic: {len(fr_preferences_data):,}")
print(f"  Total time: {total_time/3600:.2f} hours")
print(f"  Average rate: {samples_processed/total_time:.1f} samples/sec")
print(f"  Errors: {errors_count}")

print("\nJudgment Distribution:")
for judgment_type, count in judgment_stats.items():
    total = sum(judgment_stats.values())
    pct = 100 * count / total if total > 0 else 0
    print(f"  {judgment_type}: {count:,} ({pct:.1f}%)")

print("\nMethod Performance:")
for method_name, stats in method_stats.items():
    total = stats['chosen'] + stats['rejected']
    chosen_pct = 100 * stats['chosen'] / total if total > 0 else 0
    print(f"  {method_name}:")
    print(f"    Chosen: {stats['chosen']:,} ({chosen_pct:.1f}%)")
    print(f"    Rejected: {stats['rejected']:,} ({100-chosen_pct:.1f}%)")

Clearing GPU memory...
✓ GPU memory cleared

PREFERENCE DATASET GENERATION WITH GEMMA JUDGE

Configuration:
  Total samples: 20,000
  Batch size: 64
  Candidates per sample: 2
  Judge: Gemma LLM
  GPUs: 2

Language distribution:
  English to Arabic: 10,000 (50.0%)
  French to Arabic: 10,000 (50.0%)

📝 No checkpoint found. Starting from beginning...


Processing batches:   3%|▎         | 10/313 [08:34<4:16:03, 50.71s/it]


Progress (Batch 10/313):
  Samples: 640/20,000 (3.2%)
  Preference pairs: 636
    EN→AR: 285, FR→AR: 351
  Judgments: A=636, B=0, Equal=4
  Rate: 1.2 samples/sec
  ETA: 4.32 hours
  Errors: 0
  💾 Checkpoint saved


Processing batches:   6%|▋         | 20/313 [17:09<4:13:19, 51.87s/it]


Progress (Batch 20/313):
  Samples: 1,280/20,000 (6.4%)
  Preference pairs: 1,268
    EN→AR: 620, FR→AR: 648
  Judgments: A=1268, B=0, Equal=12
  Rate: 1.2 samples/sec
  ETA: 4.18 hours
  Errors: 0
  💾 Checkpoint saved


...


Progress (Batch 230/313):
  Samples: 14,720/20,000 (73.6%)
  Preference pairs: 14,610
    EN→AR: 7,290, FR→AR: 7,320
  Judgments: A=14610, B=0, Equal=110
  Rate: 1.3 samples/sec
  ETA: 1.15 hours
  Errors: 0
  💾 Checkpoint saved


Processing batches:  77%|███████▋  | 240/313 [3:21:33<1:01:35, 50.63s/it]


Progress (Batch 240/313):
  Samples: 15,360/20,000 (76.8%)
  Preference pairs: 15,244
    EN→AR: 7,609, FR→AR: 7,635
  Judgments: A=15244, B=0, Equal=116
  Rate: 1.3 samples/sec
  ETA: 1.01 hours
  Errors: 0
  💾 Checkpoint saved


Processing batches:  77%|███████▋  | 240/313 [3:22:06<1:01:28, 50.53s/it]


KeyboardInterrupt: 

In [ ]:
# ===========================
# MAIN GENERATION LOOP WITH GEMMA JUDGE AND CHECKPOINTING
# ===========================

# Clear GPU memory before starting
print("Clearing GPU memory...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()
    print("✓ GPU memory cleared")

print("\n" + "=" * 80)
print("PREFERENCE DATASET GENERATION WITH GEMMA JUDGE")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Total samples: {len(training_samples):,}")
print(f"  Batch size: {MEGA_BATCH_SIZE}")
print(f"  Candidates per sample: 2")
print(f"  Judge: Gemma LLM")
print(f"  GPUs: {NUM_GPUS}")

en_count = sum(1 for s in training_samples if s['source_lang'] == 'en')
fr_count = sum(1 for s in training_samples if s['source_lang'] == 'fr')
print(f"\nLanguage distribution:")
print(f"  English to Arabic: {en_count:,} ({100*en_count/len(training_samples):.1f}%)")
print(f"  French to Arabic: {fr_count:,} ({100*fr_count/len(training_samples):.1f}%)")
print("=" * 80)

# Output file paths
en_ar_preferences_file = OUTPUTS_DIR / "en-ar-preferences.jsonl"
fr_ar_preferences_file = OUTPUTS_DIR / "fr-ar-preferences.jsonl"

# Checkpoint file paths
checkpoint_dir = DATA_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = checkpoint_dir / "generation_checkpoint.json"
en_preferences_checkpoint = checkpoint_dir / "en_preferences_checkpoint.jsonl"
fr_preferences_checkpoint = checkpoint_dir / "fr_preferences_checkpoint.jsonl"
stats_checkpoint = checkpoint_dir / "stats_checkpoint.json"

# Load checkpoint if exists
resume_from_batch = 0
en_preferences_data = []
fr_preferences_data = []
judgment_stats = {'A is better': 0, 'B is better': 0, 'Equal': 0}
method_stats = {'high_temp': {'chosen': 0, 'rejected': 0}, 'conservative': {'chosen': 0, 'rejected': 0}}
errors_count = 0

if checkpoint_file.exists():
    print("\n⏳ Loading checkpoint...")
    try:
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
        resume_from_batch = checkpoint['last_completed_batch'] + 1
        errors_count = checkpoint.get('errors_count', 0)
        
        # Load checkpoint data files
        if en_preferences_checkpoint.exists():
            with open(en_preferences_checkpoint, 'r', encoding='utf-8') as f:
                en_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if fr_preferences_checkpoint.exists():
            with open(fr_preferences_checkpoint, 'r', encoding='utf-8') as f:
                fr_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if stats_checkpoint.exists():
            with open(stats_checkpoint, 'r') as f:
                checkpoint_stats = json.load(f)
            judgment_stats = checkpoint_stats.get('judgment_stats', judgment_stats)
            method_stats = checkpoint_stats.get('method_stats', method_stats)
        
        print(f"✓ Checkpoint loaded!")
        print(f"  Resuming from batch {resume_from_batch}")
        print(f"  EN preference pairs: {len(en_preferences_data):,}")
        print(f"  FR preference pairs: {len(fr_preferences_data):,}")
    except Exception as e:
        print(f"✗ Error loading checkpoint: {e}")
        print("  Starting from beginning...")
        resume_from_batch = 0
else:
    print("\n📝 No checkpoint found. Starting from beginning...")

num_batches = (len(training_samples) + MEGA_BATCH_SIZE - 1) // MEGA_BATCH_SIZE
start_time = time.time()
samples_processed = resume_from_batch * MEGA_BATCH_SIZE
checkpoint_interval = 10  # Save checkpoint every 10 batches

for batch_idx in tqdm(range(resume_from_batch, num_batches), desc="Processing batches", initial=resume_from_batch, total=num_batches):
    start_idx = batch_idx * MEGA_BATCH_SIZE
    end_idx = min(start_idx + MEGA_BATCH_SIZE, len(training_samples))
    batch_samples = training_samples[start_idx:end_idx]
    
    batch_sources = [s['source'] for s in batch_samples]
    batch_langs = [s['source_lang'] for s in batch_samples]
    
    try:
        # Generate 2 candidates per sample using different methods
        candidates_1 = generate_candidate_1(batch_sources, batch_langs)
        
        # Clear GPU cache between generations
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        
        candidates_2 = generate_candidate_2(batch_sources, batch_langs)
        
        # Process each sample: judge translations and create preference pair
        for sample_idx, sample in enumerate(batch_samples):
            source_text = sample['source']
            source_lang = sample['source_lang']
            
            cand_1 = candidates_1[sample_idx]
            cand_2 = candidates_2[sample_idx]
            
            # Use Gemma as judge
            try:
                chosen, rejected, chosen_method, rejected_method, judgment = judge_with_gemma(
                    source_text, source_lang, cand_1, cand_2
                )
                
                # Track statistics
                if "A is better" in judgment:
                    judgment_stats['A is better'] += 1
                elif "B is better" in judgment:
                    judgment_stats['B is better'] += 1
                else:
                    judgment_stats['Equal'] += 1
                
                method_stats[chosen_method]['chosen'] += 1
                method_stats[rejected_method]['rejected'] += 1
                
                # Create preference record
                preference_record = {
                    'source': source_text,
                    'source_lang': source_lang,
                    'chosen': chosen,
                    'rejected': rejected,
                    'chosen_method': chosen_method,
                    'rejected_method': rejected_method,
                    'judgment': judgment,
                    'candidate_1': cand_1['translation'],
                    'candidate_2': cand_2['translation'],
                    'candidate_1_method': cand_1['method'],
                    'candidate_2_method': cand_2['method']
                }
                
                # Store in appropriate list
                if source_lang == 'en':
                    en_preferences_data.append(preference_record)
                elif source_lang == 'fr':
                    fr_preferences_data.append(preference_record)
                    
            except Exception as e:
                print(f"\n⚠️  Error judging sample {sample_idx} in batch {batch_idx}: {e}")
                errors_count += 1
                continue
        
        samples_processed += len(batch_samples)
        
        # Aggressive memory management
        if batch_idx % 5 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    
    except Exception as e:
        errors_count += 1
        if errors_count <= 5:
            print(f"\n✗ Error in batch {batch_idx}: {e}")
        continue
    
    # Save checkpoint periodically
    if (batch_idx + 1) % checkpoint_interval == 0 or batch_idx == num_batches - 1:
        checkpoint_data = {
            'last_completed_batch': batch_idx,
            'samples_processed': samples_processed,
            'en_pairs_count': len(en_preferences_data),
            'fr_pairs_count': len(fr_preferences_data),
            'errors_count': errors_count,
            'timestamp': time.time()
        }
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)
        
        # Save checkpoint data files
        with open(en_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in en_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        with open(fr_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in fr_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        checkpoint_stats_data = {
            'judgment_stats': judgment_stats,
            'method_stats': method_stats
        }
        with open(stats_checkpoint, 'w') as f:
            json.dump(checkpoint_stats_data, f, indent=2)
    
    # Progress update
    if (batch_idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = samples_processed / elapsed if elapsed > 0 else 0
        remaining = (len(training_samples) - samples_processed) / rate if rate > 0 else 0
        
        total_pairs = len(en_preferences_data) + len(fr_preferences_data)
        
        print(f"\nProgress (Batch {batch_idx + 1}/{num_batches}):")
        print(f"  Samples: {samples_processed:,}/{len(training_samples):,} ({100*samples_processed/len(training_samples):.1f}%)")
        print(f"  Preference pairs: {total_pairs:,}")
        print(f"    EN→AR: {len(en_preferences_data):,}, FR→AR: {len(fr_preferences_data):,}")
        print(f"  Judgments: A={judgment_stats['A is better']}, B={judgment_stats['B is better']}, Equal={judgment_stats['Equal']}")
        print(f"  Rate: {rate:.1f} samples/sec")
        print(f"  ETA: {remaining/3600:.2f} hours")
        print(f"  Errors: {errors_count}")
        print(f"  💾 Checkpoint saved")

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("GENERATION COMPLETE")
print("=" * 80)
print(f"  Samples processed: {samples_processed:,}")
print(f"  Total preference pairs: {len(en_preferences_data) + len(fr_preferences_data):,}")
print(f"    English to Arabic: {len(en_preferences_data):,}")
print(f"    French to Arabic: {len(fr_preferences_data):,}")
print(f"  Total time: {total_time/3600:.2f} hours")
print(f"  Average rate: {samples_processed/total_time:.1f} samples/sec")
print(f"  Errors: {errors_count}")

print("\nJudgment Distribution:")
for judgment_type, count in judgment_stats.items():
    total = sum(judgment_stats.values())
    pct = 100 * count / total if total > 0 else 0
    print(f"  {judgment_type}: {count:,} ({pct:.1f}%)")

print("\nMethod Performance:")
for method_name, stats in method_stats.items():
    total = stats['chosen'] + stats['rejected']
    chosen_pct = 100 * stats['chosen'] / total if total > 0 else 0
    print(f"  {method_name}:")
    print(f"    Chosen: {stats['chosen']:,} ({chosen_pct:.1f}%)")
    print(f"    Rejected: {stats['rejected']:,} ({100-chosen_pct:.1f}%)")


Clearing GPU memory...
✓ GPU memory cleared

PREFERENCE DATASET GENERATION WITH GEMMA JUDGE

Configuration:
  Total samples: 20,000
  Batch size: 64
  Candidates per sample: 2
  Judge: Gemma LLM
  GPUs: 2

Language distribution:
  English to Arabic: 10,000 (50.0%)
  French to Arabic: 10,000 (50.0%)

📝 No checkpoint found. Starting from beginning...
✓ GPU memory cleared

PREFERENCE DATASET GENERATION WITH GEMMA JUDGE

Configuration:
  Total samples: 20,000
  Batch size: 64
  Candidates per sample: 2
  Judge: Gemma LLM
  GPUs: 2

Language distribution:
  English to Arabic: 10,000 (50.0%)
  French to Arabic: 10,000 (50.0%)

📝 No checkpoint found. Starting from beginning...


Processing batches:   0%|          | 0/313 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Processing batches:   3%|▎         | 8/313 [11:35<7:21:38, 86.88s/it]


KeyboardInterrupt: 

## Choose Generation Mode

Select between sequential (safer) or batch judging (much faster).

In [ ]:
# ===========================
# CONFIGURATION: CHOOSE JUDGING MODE
# ===========================

USE_BATCH_JUDGING = True  # Set to True for 8-10x speedup, False for sequential (safer)

if USE_BATCH_JUDGING:
    print("✅ Using BATCH JUDGING mode (8-10x faster!)")
    print("   Judges multiple samples at once")
    print("   Estimated speedup: 8-10x")
else:
    print("⚠️  Using SEQUENTIAL JUDGING mode (slower but safer)")
    print("   Judges one sample at a time")
    print("   This will take significantly longer")

print(f"\nWith current settings ({len(training_samples):,} samples):")
if USE_BATCH_JUDGING:
    estimated_hours = (len(training_samples) / MEGA_BATCH_SIZE) * (16 + 2) / 3600
    print(f"   Estimated time: ~{estimated_hours:.2f} hours")
else:
    estimated_hours = (len(training_samples) / MEGA_BATCH_SIZE) * (16 + MEGA_BATCH_SIZE * 0.3) / 3600
    print(f"   Estimated time: ~{estimated_hours:.2f} hours")

## 🚀 OPTIMIZED Generation Loop (Use This!)

This version:
- ✅ Uses **batch judging** (8-10x faster)
- ✅ Checkpoints **every 100 preference pairs**
- ✅ Fixed tokenization warnings
- ✅ Better progress reporting

In [ ]:
# ===========================
# OPTIMIZED MAIN LOOP WITH BATCH JUDGING + FREQUENT CHECKPOINTS
# ===========================

# Clear GPU memory before starting
print("Clearing GPU memory...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()
    print("✓ GPU memory cleared")

print("\n" + "=" * 80)
print("OPTIMIZED PREFERENCE DATASET GENERATION")
print("=" * 80)
print("✅ Using BATCH JUDGING (8-10x faster!)")
print("✅ Checkpointing every 100 preference pairs")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Total samples: {len(training_samples):,}")
print(f"  Batch size: {MEGA_BATCH_SIZE}")
print(f"  Candidates per sample: 2")
print(f"  Judge: Gemma LLM (batch mode)")
print(f"  GPUs: {NUM_GPUS}")

en_count = sum(1 for s in training_samples if s['source_lang'] == 'en')
fr_count = sum(1 for s in training_samples if s['source_lang'] == 'fr')
print(f"\nLanguage distribution:")
print(f"  English to Arabic: {en_count:,} ({100*en_count/len(training_samples):.1f}%)")
print(f"  French to Arabic: {fr_count:,} ({100*fr_count/len(training_samples):.1f}%)")
print("=" * 80)

# Output file paths
en_ar_preferences_file = OUTPUTS_DIR / "en-ar-preferences.jsonl"
fr_ar_preferences_file = OUTPUTS_DIR / "fr-ar-preferences.jsonl"

# Checkpoint file paths
checkpoint_dir = DATA_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = checkpoint_dir / "generation_checkpoint.json"
en_preferences_checkpoint = checkpoint_dir / "en_preferences_checkpoint.jsonl"
fr_preferences_checkpoint = checkpoint_dir / "fr_preferences_checkpoint.jsonl"
stats_checkpoint = checkpoint_dir / "stats_checkpoint.json"

# Load checkpoint if exists
resume_from_batch = 0
en_preferences_data = []
fr_preferences_data = []
judgment_stats = {'A is better': 0, 'B is better': 0, 'Equal': 0}
method_stats = {'high_temp': {'chosen': 0, 'rejected': 0}, 'conservative': {'chosen': 0, 'rejected': 0}}
errors_count = 0
last_checkpoint_count = 0  # Track when last checkpoint was saved

if checkpoint_file.exists():
    print("\n⏳ Loading checkpoint...")
    try:
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
        resume_from_batch = checkpoint['last_completed_batch'] + 1
        errors_count = checkpoint.get('errors_count', 0)
        last_checkpoint_count = checkpoint.get('last_checkpoint_count', 0)
        
        # Load checkpoint data files
        if en_preferences_checkpoint.exists():
            with open(en_preferences_checkpoint, 'r', encoding='utf-8') as f:
                en_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if fr_preferences_checkpoint.exists():
            with open(fr_preferences_checkpoint, 'r', encoding='utf-8') as f:
                fr_preferences_data = [json.loads(line) for line in f if line.strip()]
        
        if stats_checkpoint.exists():
            with open(stats_checkpoint, 'r') as f:
                checkpoint_stats = json.load(f)
            judgment_stats = checkpoint_stats.get('judgment_stats', judgment_stats)
            method_stats = checkpoint_stats.get('method_stats', method_stats)
        
        print(f"✓ Checkpoint loaded!")
        print(f"  Resuming from batch {resume_from_batch}")
        print(f"  EN preference pairs: {len(en_preferences_data):,}")
        print(f"  FR preference pairs: {len(fr_preferences_data):,}")
    except Exception as e:
        print(f"✗ Error loading checkpoint: {e}")
        print("  Starting from beginning...")
        resume_from_batch = 0
        last_checkpoint_count = 0
else:
    print("\n📝 No checkpoint found. Starting from beginning...")

num_batches = (len(training_samples) + MEGA_BATCH_SIZE - 1) // MEGA_BATCH_SIZE
start_time = time.time()
samples_processed = resume_from_batch * MEGA_BATCH_SIZE
checkpoint_every_n_pairs = 100  # Save checkpoint every 100 preference pairs
batch_times = []  # Track batch times for better ETA

for batch_idx in tqdm(range(resume_from_batch, num_batches), desc="Processing batches", initial=resume_from_batch, total=num_batches):
    batch_start_time = time.time()
    
    start_idx = batch_idx * MEGA_BATCH_SIZE
    end_idx = min(start_idx + MEGA_BATCH_SIZE, len(training_samples))
    batch_samples = training_samples[start_idx:end_idx]
    
    batch_sources = [s['source'] for s in batch_samples]
    batch_langs = [s['source_lang'] for s in batch_samples]
    
    try:
        # Generate 2 candidates per sample using different methods
        candidates_1 = generate_candidate_1(batch_sources, batch_langs)
        
        # Clear GPU cache between generations
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        
        candidates_2 = generate_candidate_2(batch_sources, batch_langs)
        
        # BATCH JUDGING (MUCH FASTER!)
        try:
            judgments = judge_with_gemma_batch(batch_sources, batch_langs, candidates_1, candidates_2)
            
            # Process results
            for sample_idx, sample in enumerate(batch_samples):
                source_text = sample['source']
                source_lang = sample['source_lang']
                cand_1 = candidates_1[sample_idx]
                cand_2 = candidates_2[sample_idx]
                
                chosen, rejected, chosen_method, rejected_method, judgment = judgments[sample_idx]
                
                # Track statistics
                if "A is better" in judgment:
                    judgment_stats['A is better'] += 1
                elif "B is better" in judgment:
                    judgment_stats['B is better'] += 1
                else:
                    judgment_stats['Equal'] += 1
                
                method_stats[chosen_method]['chosen'] += 1
                method_stats[rejected_method]['rejected'] += 1
                
                # Create preference record
                preference_record = {
                    'source': source_text,
                    'source_lang': source_lang,
                    'chosen': chosen,
                    'rejected': rejected,
                    'chosen_method': chosen_method,
                    'rejected_method': rejected_method,
                    'judgment': judgment,
                    'candidate_1': cand_1['translation'],
                    'candidate_2': cand_2['translation'],
                    'candidate_1_method': cand_1['method'],
                    'candidate_2_method': cand_2['method']
                }
                
                # Store in appropriate list
                if source_lang == 'en':
                    en_preferences_data.append(preference_record)
                elif source_lang == 'fr':
                    fr_preferences_data.append(preference_record)
                    
        except Exception as e:
            print(f"\n⚠️  Error in batch judging: {e}")
            errors_count += 1
        
        samples_processed += len(batch_samples)
        
        # Track batch time
        batch_time = time.time() - batch_start_time
        batch_times.append(batch_time)
        if len(batch_times) > 10:  # Keep last 10 batch times
            batch_times.pop(0)
        
        # Aggressive memory management
        if batch_idx % 5 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    
    except Exception as e:
        errors_count += 1
        if errors_count <= 5:
            print(f"\n✗ Error in batch {batch_idx}: {e}")
        continue
    
    # CHECKPOINT EVERY 100 PREFERENCE PAIRS
    total_pairs = len(en_preferences_data) + len(fr_preferences_data)
    if total_pairs >= last_checkpoint_count + checkpoint_every_n_pairs or batch_idx == num_batches - 1:
        checkpoint_data = {
            'last_completed_batch': batch_idx,
            'samples_processed': samples_processed,
            'en_pairs_count': len(en_preferences_data),
            'fr_pairs_count': len(fr_preferences_data),
            'errors_count': errors_count,
            'last_checkpoint_count': total_pairs,
            'timestamp': time.time()
        }
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)
        
        # Save checkpoint data files
        with open(en_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in en_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        with open(fr_preferences_checkpoint, 'w', encoding='utf-8') as f:
            for item in fr_preferences_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        checkpoint_stats_data = {
            'judgment_stats': judgment_stats,
            'method_stats': method_stats
        }
        with open(stats_checkpoint, 'w') as f:
            json.dump(checkpoint_stats_data, f, indent=2)
        
        last_checkpoint_count = total_pairs
    
    # DETAILED PROGRESS UPDATE
    if (batch_idx + 1) % 5 == 0 or batch_idx == 0:  # Update every 5 batches
        elapsed = time.time() - start_time
        avg_batch_time = sum(batch_times) / len(batch_times) if batch_times else batch_time
        rate = samples_processed / elapsed if elapsed > 0 else 0
        remaining_batches = num_batches - (batch_idx + 1)
        remaining_time = remaining_batches * avg_batch_time
        
        total_pairs = len(en_preferences_data) + len(fr_preferences_data)
        
        print(f"\n{'='*60}")
        print(f"Progress (Batch {batch_idx + 1}/{num_batches}):")
        print(f"  Samples: {samples_processed:,}/{len(training_samples):,} ({100*samples_processed/len(training_samples):.1f}%)")
        print(f"  Preference pairs: {total_pairs:,}")
        print(f"    EN→AR: {len(en_preferences_data):,}, FR→AR: {len(fr_preferences_data):,}")
        print(f"  Judgments: A={judgment_stats['A is better']}, B={judgment_stats['B is better']}, Equal={judgment_stats['Equal']}")
        print(f"  Batch time: {batch_time:.1f}s (avg: {avg_batch_time:.1f}s)")
        print(f"  Rate: {rate:.2f} samples/sec")
        print(f"  ETA: {remaining_time/3600:.2f} hours ({remaining_time/60:.1f} min)")
        print(f"  Errors: {errors_count}")
        if total_pairs >= last_checkpoint_count:
            print(f"  💾 Checkpoint saved at {total_pairs} pairs")
        print(f"{'='*60}")

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("GENERATION COMPLETE")
print("=" * 80)
print(f"  Samples processed: {samples_processed:,}")
print(f"  Total preference pairs: {len(en_preferences_data) + len(fr_preferences_data):,}")
print(f"    English to Arabic: {len(en_preferences_data):,}")
print(f"    French to Arabic: {len(fr_preferences_data):,}")
print(f"  Total time: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")
print(f"  Average rate: {samples_processed/total_time:.2f} samples/sec")
if batch_times:
    avg_batch = sum(batch_times) / len(batch_times)
    print(f"  Average batch time: {avg_batch:.1f}s")
print(f"  Errors: {errors_count}")

print("\nJudgment Distribution:")
for judgment_type, count in judgment_stats.items():
    total = sum(judgment_stats.values())
    pct = 100 * count / total if total > 0 else 0
    print(f"  {judgment_type}: {count:,} ({pct:.1f}%)")

print("\nMethod Performance:")
for method_name, stats in method_stats.items():
    total = stats['chosen'] + stats['rejected']
    chosen_pct = 100 * stats['chosen'] / total if total > 0 else 0
    print(f"  {method_name}:")
    print(f"    Chosen: {stats['chosen']:,} ({chosen_pct:.1f}%)")
    print(f"    Rejected: {stats['rejected']:,} ({100-chosen_pct:.1f}%)")

print("\n" + "=" * 80)
print("✅ Use this optimized version for much faster generation!")
print("=" * 80)


## Save Synthetic Dataset

In [ ]:
# Save datasets to separate files
print(f"Saving datasets...\n")

# Save EN-AR preferences
print(f"Saving {len(en_preferences_data):,} EN-AR preference pairs to {en_ar_preferences_file.name}...")
with open(en_ar_preferences_file, 'w', encoding='utf-8') as f:
    for item in en_preferences_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"  ✓ Saved")

# Save FR-AR preferences
print(f"Saving {len(fr_preferences_data):,} FR-AR preference pairs to {fr_ar_preferences_file.name}...")
with open(fr_ar_preferences_file, 'w', encoding='utf-8') as f:
    for item in fr_preferences_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"  ✓ Saved")

# Calculate statistics
all_pairs = en_preferences_data + fr_preferences_data

# Prepare statistics
stats = {
    'total_preference_pairs': len(all_pairs),
    'en_pairs': len(en_preferences_data),
    'fr_pairs': len(fr_preferences_data),
    'language_breakdown': {
        'english': {
            'pairs': len(en_preferences_data),
            'percentage': 100 * len(en_preferences_data) / len(all_pairs) if all_pairs else 0
        },
        'french': {
            'pairs': len(fr_preferences_data),
            'percentage': 100 * len(fr_preferences_data) / len(all_pairs) if all_pairs else 0
        }
    },
    'judgment_distribution': {
        'a_better': judgment_stats['A is better'],
        'b_better': judgment_stats['B is better'],
        'equal': judgment_stats['Equal']
    },
    'method_performance': {
        method_name: {
            'chosen_count': stats_data['chosen'],
            'rejected_count': stats_data['rejected'],
            'chosen_percentage': 100 * stats_data['chosen'] / (stats_data['chosen'] + stats_data['rejected']) if (stats_data['chosen'] + stats_data['rejected']) > 0 else 0
        }
        for method_name, stats_data in method_stats.items()
    },
    'generation_config': {
        'num_candidates': NUM_CANDIDATES,
        'max_new_tokens': MAX_NEW_TOKENS,
        'judge_model': 'Gemma',
        'batch_size': MEGA_BATCH_SIZE
    }
}

stats_path = OUTPUTS_DIR / "generation_stats.json"
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)

# Print summary
print("\n" + "=" * 80)
print("DATASET STATISTICS")
print("=" * 80)

print(f"\nPreference Pairs:")
print(f"  Total pairs: {stats['total_preference_pairs']:,}")
print(f"  English to Arabic: {stats['en_pairs']:,} ({stats['language_breakdown']['english']['percentage']:.1f}%)")
print(f"  French to Arabic: {stats['fr_pairs']:,} ({stats['language_breakdown']['french']['percentage']:.1f}%)")

print(f"\nJudgment Distribution:")
print(f"  A is better: {stats['judgment_distribution']['a_better']:,}")
print(f"  B is better: {stats['judgment_distribution']['b_better']:,}")
print(f"  Equal: {stats['judgment_distribution']['equal']:,}")

print(f"\nMethod Performance:")
for method_name, perf in stats['method_performance'].items():
    print(f"  {method_name}:")
    print(f"    Chosen: {perf['chosen_count']:,} ({perf['chosen_percentage']:.1f}%)")
    print(f"    Rejected: {perf['rejected_count']:,} ({100-perf['chosen_percentage']:.1f}%)")

print(f"\nGenerated files:")
print(f"  - {en_ar_preferences_file.name}")
print(f"  - {fr_ar_preferences_file.name}")
print(f"  - {stats_path.name}")

print(f"\nStatistics saved to {stats_path}")
print("=" * 80)


Saving datasets...

Saving 10,000 EN-AR candidates to english_arabic_candidates.jsonl...
  ✓ Saved
Saving 10,000 FR-AR candidates to french_arabic_candidates.jsonl...
  ✓ Saved
Saving 44,324 EN-AR preference pairs to en-ar-preferences.jsonl...
  ✓ Saved
Saving 46,908 FR-AR preference pairs to fr-ar-preferences.jsonl...
  ✓ Saved
Saving 46,908 FR-AR preference pairs to fr-ar-preferences.jsonl...
  ✓ Saved

DATASET STATISTICS

Candidates:
  English to Arabic: 10,000
  French to Arabic: 10,000
  Total: 20,000

Preference Pairs:
  Total pairs: 91,232
  Average margin: 0.0633
  Average chosen score: 0.9166

Language Breakdown:
  English to Arabic:
    Candidates: 10,000
    Preference pairs: 44,324
    Avg margin: 0.0549
    Avg score: 0.9224
  French to Arabic:
    Candidates: 10,000
    Preference pairs: 46,908
    Avg margin: 0.0712
    Avg score: 0.9111

Method Breakdown (Preference Pairs):
  temperature:
    Chosen: 24,553
    Rejected: 24,405
    Avg score: 0.8820
  top_k:
    Chosen:

## Sample Preference Pairs

In [ ]:
# Display sample generated data
print("Sample Generated Preference Pairs\n")
print("=" * 80)

# Show sample preferences from EN and FR
en_examples = random.sample(en_preferences_data, min(3, len(en_preferences_data)))
fr_examples = random.sample(fr_preferences_data, min(2, len(fr_preferences_data)))

for i, item in enumerate(en_examples + fr_examples, 1):
    lang_label = 'EN→AR' if item['source_lang'] == 'en' else 'FR→AR'
    print(f"\nExample {i}: {lang_label}")
    print(f"Source: {item['source'][:100]}")
    
    print(f"\nCandidate 1 ({item['candidate_1_method']}):")
    print(f"  {item['candidate_1'][:100]}")
    
    print(f"\nCandidate 2 ({item['candidate_2_method']}):")
    print(f"  {item['candidate_2'][:100]}")
    
    print(f"\nGemma's Judgment: {item['judgment']}")
    print(f"✓ Chosen ({item['chosen_method']}): {item['chosen'][:100]}")
    print(f"✗ Rejected ({item['rejected_method']}): {item['rejected'][:100]}")
    print("-" * 80)

# Summary of Gemma's preferences
print("\n" + "=" * 80)
print("GEMMA JUDGE SUMMARY")
print("=" * 80)

all_pairs = en_preferences_data + fr_preferences_data
total_judgments = len(all_pairs)

print(f"\nTotal judgments made: {total_judgments:,}")

print(f"\nJudgment breakdown:")
for judgment_type, count in judgment_stats.items():
    pct = 100 * count / total_judgments if total_judgments > 0 else 0
    print(f"  {judgment_type}: {count:,} ({pct:.1f}%)")

print(f"\nMethod preferences:")
for method_name, stats in method_stats.items():
    total = stats['chosen'] + stats['rejected']
    chosen_pct = 100 * stats['chosen'] / total if total > 0 else 0
    print(f"  {method_name}:")
    print(f"    Chosen: {stats['chosen']:,} ({chosen_pct:.1f}%)")
    print(f"    Rejected: {stats['rejected']:,} ({100-chosen_pct:.1f}%)")

print("\n" + "=" * 80)


Sample Generated Data

SAMPLE TRANSLATION CANDIDATES (4 methods per source)

Example 1: EN→AR
Source: Implementation. Under a mandate generally deriving from Article 98 of the Charter of the United Nati
Candidates:
  1. [temperature]: التنفيذ - بموجب ولاية مستمدة عموما من أحكام المادة 98 من ميثاق الأمم المتحدة، والموافقة الصريحة أو ا
  2. [top_k]: التنفيذ - بموجب ولاية مستمدة بشكل عام من المادة 98 من ميثاق الأمم المتحدة، والموافقة الصريحة أو الضم
  3. [nucleus]: التنفيذ - في إطار ولاية مستمدة عموما من المادة 98 من ميثاق الأمم المتحدة، والموافقة الصريحة أو الضمن
  4. [greedy]: التنفيذ - بموجب ولاية مستمدة عموما من المادة 98 من ميثاق الأمم المتحدة، والموافقة الصريحة أو الضمنية

Example 2: EN→AR
Source: "In this respect, a message from the President of Maldives, Maumoon Abdul Gayoom, was read to the me
Candidates:
  1. [temperature]: "وفي هذا الصدد، تم قراءة رسالة أثناء الاجتماع من رئيس جمهورية المالديف، رئيس جمهورية المالديف، محمد 
  2. [top_k]: "في هذا الصدد، تم قراءة رسالة من رئيس ملدي

## Next Step

Proceed to **notebook 2** to train the reward model using this synthetic preference data.